# TAREFA T1
Lista de variáveis: 

Coeficiente de segurança contra falha por flexão: ${\gamma}_{s} = 2.0$ \
Coeficiente de segurança contra falha por cisalhamento: ${\gamma}_{\tau} = 3.0$ \
Índide de confiabilidade alvo contra falha por flexão: ${\beta}_{sT} = 3.0$ \
Índide de confiabilidade alvo contra falha por cisalhamento: ${\beta}_{\tau T} = 4.0$ \
Custo de falha por flexão: ${c}_{s} = 3.0$ \
Custo de falha por cisalhamento: ${c}_{\tau} = 7.0$ \
Tensão normal devido ao momento fletor: $s = \frac{6M}{bh^2}$\
Tensão cisalhante média: ${\tau} = \frac{3V}{2bh}$ \
Tensão de escoamento em flexão (MPa): $N(\mu_S; \sigma_S) = N(20;2)kNm$ \
Tensão de escoamento em cisalhamento (MPa): $N(\mu_\tau; \sigma_\tau) = N(4;0.4)kNm$ \
Momento solicitante (kNm): $N(\mu_M; \sigma_M) = N(40;8)kNm$ \
Cortante solicitante (kN): $N(\mu_V; \sigma_V) = N(150;30)kNm$ \
Restrição geométrica: $h \le 2b$ 



In [ ]:
import sympy as sp

# Declarando as variáveis
b, h, mu_M, gamma_S, mu_S, mu_V, f, gamma_tau, mu_tau, = sp.symbols('b, h, mu_M, gamma_S, mu_S, mu_V, f, gamma_tau, mu_tau')
mu_M = 40
gamma_S = 2.0
mu_S = 20.0
mu_V = 150.0
gamma_tau = 3.0
mu_tau = 4.0

# Funções 
f = b*h #Cost function
g_s = (6*mu_M*gamma_S) - (mu_S*b*h**2) # Restrição limite de flexão
g_tau = (3*mu_V*gamma_tau) - (2*mu_tau*b*h) # Restrição limite de esforço cortante
g_a = h-(2*b) # Restrição limite geometrica

In [ ]:
# Verificação da função de restrição de limite de flexão g_s

variables_order_gs = list(ordered(g_s.free_symbols))
hessian_gs = sp.hessian(g_s, variables_order_gs)
auto_valores_hgs = hessian_gs.eigenvals()
print(hessian_gs)
print(auto_valores_hgs)
print("Como a Hessiana não é positiva semi-definida, a restrição g_s não é convexa.")

Agora precisamos verificar as condições de convexidade das equações apresentadas:

In [ ]:
# Verificação da função de restrição de limite de cisalhamento g_tau

variables_order_gtau = list(ordered(g_tau.free_symbols))
hessian_gtau = sp.hessian(g_tau, variables_order_gtau)
auto_valores_hgtau = hessian_gtau.eigenvals()
print(hessian_gtau)
print(auto_valores_hgtau)
print("Como a Hessiana não é positiva semi-definida, a restrição g_tau não é convexa.")

In [ ]:
# Verificação da função custo

variables_order_f = list(ordered(f.free_symbols))
hessian_f = sp.hessian(f, variables_order_f)
auto_valores_f = hessian_f.eigenvals()
print(hessian_f)
print(auto_valores_f)
print("Como a Hessiana não é positiva semi-definida, a restrição f não é convexa.")

O resultado do teste de convexidade mostra que não há como garantir a existencia de um minimo global para o problema.\
Agora seguimos com a verificação das condições necessárias de KKT.\
Para isso, escrevemos a função lagrangiana do problema: 
$$
L = f + u_1 (g_s + s_1^2) + u_2 (g_\tau + s_2^2) + u_3 (h - 2b + s_3^2)
$$


In [ ]:
# Vamos escrever a função lagrangiana do problema
u_1, u_2, u_3, s_1, s_2, s_3 = sp.symbols('u_1, u_2, u_3, s_1, s_2, s_3')

L = f + u_1 * (g_s + s_1**2) + u_2 * (g_tau + s_2**2) + u_3 * (g_a + s_3**2) # Função Lagrangiana

# Agora vamos computar todas as derivadas parciais da Lagrangiana
dL_db = sp.diff(L, b)
dL_dh = sp.diff(L, h)
dL_du1 = sp.diff(L, u_1)
dL_du2 = sp.diff(L, u_2)
dL_du3 = sp.diff(L, u_3)
dL_ds1 = sp.diff(L, s_1)
dL_ds2 = sp.diff(L, s_2)
dL_ds3 = sp.diff(L, s_3)

In [ ]:
# Caso 1 (u_1 = u_2 = u_3 = 0)

valores_u = {u_1: 0, u_2: 0, u_3: 0}
dL_db_new = dL_db.subs(valores_u)
dL_dh_new = dL_dh.subs(valores_u)
sp.solve([dL_db_new, dL_dh_new], (h,b))

In [ ]:
# Caso 2 (u_1 = u_2 = 0, s_3 = 0)

valores = {u_1: 0, u_2: 0, s_3: 0}
dL_db_new = dL_db.subs(valores)
dL_dh_new = dL_dh.subs(valores)
sp.solve([dL_db_new, dL_dh_new], (h,b))

In [ ]:
# Caso 3 (u_1 = u_2 = 0, s_3 = 0)

valores = {u_1: 0, u_2: 0, s_3: 0}
dL_db_new = dL_db.subs(valores)
dL_dh_new = dL_dh.subs(valores)
sp.solve([dL_db_new, dL_dh_new], (h,b))

# TAREFA T2
Resolução da Tarefa 1 utilizando os métodos de otimização diretos.



1) Abordagem DDO \
Temos que a abordagem DDO do problema de minimização da seção da viga submetida a momento fletor e esforço cortante, pode ser escrita como:

$$
\text{determine: } \mathbf{d^*} = \{b^*, h^*\},\\
\text{que minimiza: } f(\mathbf{d}) = bh , \\
\text {sujeito a: } g_s(\mathbf{d}) = 6\mu_M\gamma_S - \mu_Sbh^2 \le 0 , \\
                    g_{\tau(\mathbf{d})} = 3\mu_V\gamma_\tau - 2\mu_{\tau}bh \le 0, \\
                    h - 2b \le 0.

$$


In [24]:
from scipy.optimize import minimize

# Constant variables
mu_M = 40
gamma_S = 2.0
mu_S = 20.0
mu_V = 150.0
gamma_tau = 3.0
mu_tau = 4.0

# Cost function
f = lambda x: (x[0] * x[1])

# Constraints
const = ({'type' : 'ineq', 'fun': lambda x: -((6 * mu_M * gamma_S) - (mu_S * x[0] * (x[1]**2)))}, 
         {'type' : 'ineq', 'fun': lambda x: -((6 * mu_V * gamma_tau) - (2 * mu_tau * x[0] * x[1]))},
         {'type' : 'ineq', 'fun': lambda x: -(x[1] - (2 * x[0]))})

bnds = ((0,None), (0, None))

result = minimize(f, (100, 100), method='SLSQP', bounds=bnds, constraints=const)
print(result)


     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 337.50000000017036
           x: [ 1.361e+01  2.479e+01]
         nit: 10
         jac: [ 2.479e+01  1.361e+01]
        nfev: 27
        njev: 9
 multipliers: [ 0.000e+00  1.250e-01  0.000e+00]


2) Abordagem RBDO \
Temos que a abordagem RBDO do problema de minimização da seção da viga submetida a momento fletor e esforço cortante, pode ser escrita como:

$$
\text{determine: } \mathbf{d^*} = \{b^*, h^*\},\\
\text{que minimiza: } f(\mathbf{d}) = bh , \\
\text {sujeito a: } \Beta_T - \Beta_S \le 0 , \\
                    \Beta_T - \Beta_\tau \le 0, \\
                    h - 2b \le 0.

$$


In [36]:
from scipy.optimize import minimize
import numpy as np

# Constant variables
mu_M = 40
gamma_S = 2.0
mu_S = 20.0
mu_V = 150.0
gamma_tau = 3.0
mu_tau = 4.0
beta_s_tar = 3
beta_tau_tar = 4
sigma_s = 2
sigma_m = 8
sigma_tau = 0.4
sigma_v = 30

# Cost function
f = lambda x: (x[0] * x[1])

# Constraints
g_1 = lambda x: -(beta_s_tar - ((x[0] * (x[1]**2) * mu_S) - (6 * mu_M)) / (np.sqrt(((x[0]**2) * (x[1]**4) * (sigma_s**2)) + ((6**2) * (sigma_m**2))))) 
g_2 = lambda x: -(beta_tau_tar - ((2 * x[0] * x[1] * mu_tau) - (3 * mu_V)) / (np.sqrt((4 * (x[0]**2) * (x[1]**2) * (sigma_tau**2)) + ((3**2) * (sigma_v**2)))))
g_3 = lambda x: -(x[1] - (2 * x[0]))

const = ({'type' : 'ineq', 'fun': g_1}, 
         {'type' : 'ineq', 'fun': g_2 },
         {'type' : 'ineq', 'fun': g_3})

bnds = ((0,None), (0, None))

result = minimize(f, (10, 10), method='SLSQP', bounds=bnds, constraints=const)
print(result)


     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 122.89449915219538
           x: [ 1.109e+01  1.109e+01]
         nit: 5
         jac: [ 1.109e+01  1.109e+01]
        nfev: 16
        njev: 5
 multipliers: [ 0.000e+00  2.363e+01  0.000e+00]


# Tarefa 3

1) Resolução da rótula plástica utilizando RIA

In [21]:
import pystra as ra
from scipy.optimize import minimize

d_0 = 1.0
d_list = [d_0]
beta_t = 2.5
beta_list = []
tol = 1e-3
f_list = []
y_list = []

# FORM
def FORM(d_k):
            
    options = ra.AnalysisOptions()
    #options.setPrintOutput(True)

    limit_state_func_FORM = ra.LimitState(lambda X1,X2,X3: (d_k * (X1 * X2)) - X3)
    stochastic_model = ra.StochasticModel()

    # Define random variables
    stochastic_model.addVariable(ra.Normal("X1", 40, 5))
    stochastic_model.addVariable(ra.Normal("X2", 50, 2.5))
    stochastic_model.addVariable(ra.Normal("X3", 1000, 200))

    # Perform FORM analysis
    Analysis = ra.Form(
        analysis_options=options,
        stochastic_model=stochastic_model,
        limit_state=limit_state_func_FORM,
    )
    
    Analysis.run()
    # More detailed output
    beta  = Analysis.getBeta()
    ponto_projeto = Analysis.getDesignPoint()

    return beta, ponto_projeto

obj_func = lambda d: d

def beta_d (d):
            beta_d = FORM(d)[0]
            return beta_d

def y_projeto (d):
            y_projeto = FORM(d)[1]
            return y_projeto

def const (d):
            return beta_d(d) - beta_t

for k in range(4):
    d_k = d_list[k]
    
    # Cálculos e Otimização
    beta_list.append(beta_d(d_k))
    y_list.append(y_projeto(d_k))
    f_list.append(obj_func(d_k))
    
    cons = ({'type': 'ineq', 'fun': const})
    res = minimize(obj_func, 1, method='SLSQP', constraints=cons)
    
    d_list.append(res.x) 

    # Verificação de convergência 
    if k > 0:

        c_1 = abs(f_list[k] - f_list[k-1])
        c_2 = abs((d_list[k] - d_list[k-1]) / d_list[k-1])
        c_3 = abs((y_list[k] - y_list[k-1]) / d_list[k-1])
            
        if (c_1 <= tol).all() and (c_2 <= tol).all() and (c_3 <= tol).all():
            print(f"Convergiu na iteração {k}")
            break

d_list_final = d_list[:-1]
print(d_list_final)
print(beta_list)


Convergiu na iteração 2
[1.0, array([0.88176445]), array([0.88176445])]
[np.float64(3.049073520235253), np.float64(2.499999986045597), np.float64(2.499999986045597)]
